# Stock Price Prediction with LSTM

In this exercise we use an **LSTM** to predict Apple stock prices from historical data. We also discuss the inherent limitations of applying machine learning to stock prediction.

**Dataset:** `apple_stock.csv` -- Daily closing prices of Apple (AAPL), January 2015 to December 2023 (2,264 trading days). The prices are adjusted for splits and dividends; the file has a single `Close` column and two extra header rows from its export.

**Learning objectives:**
1. Load and visualize historical stock price data.
2. Prepare sequential data for LSTM prediction, with the scaler fitted on the training days only.
3. Build and train an LSTM model.
4. Evaluate predictions on a held-out test period against persistence (yesterday's close) on the same days.
5. Evaluate daily changes, not only price levels, and understand the limitations of ML-based stock prediction.

**Evaluation design.** Windows are split by target day; the last 20% form the test period, used once at the end. The number of epochs is fixed in advance, not tuned; to tune it, carve a validation period out of the training days, never out of the test days. This is a method demonstration, not evidence of trading value. Runtime: about 1.5 minutes on an Apple M3 Max laptop CPU; allow up to twice as long on older laptops.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# YOUR CODE HERE
# Hint: Load apple_stock.csv, skipping the two extra header rows of the export:
#   df = pd.read_csv("apple_stock.csv", skiprows=[1, 2]).rename(columns={"Price": "Date"})
#   df["Date"] = pd.to_datetime(df["Date"])
# Print shape, columns, and head()

In [ ]:
# YOUR CODE HERE
# Hint: df.describe()

In [ ]:
# YOUR CODE HERE
# Hint: Print the date range and total number of trading days

---
## 3. Visualize Stock Price Over Time

In [ ]:
# YOUR CODE HERE
# Hint: Create 2 subplots:
# Top: Close price over time (x = df["Date"])
# Bottom: daily change in the close price, df["Close"].diff() (the file has no volume column)

---
## 4. Prepare Data: Close Price, Normalize, Create Sequences

Each window belongs to the period that contains its **target** day: the first 80% of windows for training, the last 20% for the test. Test windows may read the preceding 60 observed days as inputs, which is what a deployed forecast sees, but no training target lies in the test period. The scaler is fitted on the days used by the training windows only; test prices above the training range then map above 1, which is expected.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Extract close prices: close_prices = df["Close"].values.reshape(-1, 1).astype(np.float32)
# 2. Normalize with MinMaxScaler(feature_range=(0, 1)), fitted on the TRAINING days only:
#       SEQ_LENGTH = 60
#       train_sequence_count = int((len(close_prices) - SEQ_LENGTH) * 0.8)
#       training_raw_end = SEQ_LENGTH + train_sequence_count
#       scaler.fit(close_prices[:training_raw_end])
#       close_scaled = scaler.transform(close_prices)   # all days, same fitted scaler

In [ ]:
def create_sequences(data, seq_length):
    """Create sliding-window sequences for time series prediction."""
    # YOUR CODE HERE
    # Hint: Same sliding window pattern as before
    # X[i] = data[i : i + seq_length]
    # y[i] = data[i + seq_length]
    pass


SEQ_LENGTH = 60  # use past 60 trading days (~3 months) to predict next day

X, y = create_sequences(close_scaled, SEQ_LENGTH)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Chronological split at 80% of the windows: split = int(len(X) * 0.8)
#    (check: split + SEQ_LENGTH == training_raw_end, the days the scaler saw)
# 2. Convert to PyTorch tensors
# 3. Create DataLoader with BATCH_SIZE=32

---
## 5. Build LSTM Model

In [ ]:
class StockLSTM(nn.Module):
    """LSTM for stock price prediction."""

    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        # YOUR CODE HERE
        # Hint:
        # 1. self.lstm = nn.LSTM(..., batch_first=True, dropout=dropout)
        # 2. self.fc = nn.Sequential(Linear(hidden_size, 32), ReLU(), Linear(32, 1))
        pass

    def forward(self, x):
        # YOUR CODE HERE
        # Hint: LSTM -> last time step -> FC
        pass


model = StockLSTM().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 6. Train the Model

`NUM_EPOCHS = 100` is fixed in advance; it is not chosen by looking at the test days.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. criterion = nn.MSELoss(), optimizer = optim.Adam(..., lr=1e-3)
# 2. Train for NUM_EPOCHS=100
# 3. Standard training loop with loss tracking
# 4. Print every 20 epochs

---
## 7. Plot Training Loss

In [ ]:
# YOUR CODE HERE
# Hint: plt.plot(train_losses)

---
## 8. Predict Future Prices and Plot Against Actual

This is the final evaluation: the test days are used once, for the model trained above.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. model.eval() + torch.no_grad() to get predictions
# 2. Inverse transform with scaler.inverse_transform()
# 3. Compute RMSE and MAE

In [ ]:
# YOUR CODE HERE
# Hint: Plot full price history with prediction overlay
# Show the train/test split line

In [ ]:
# YOUR CODE HERE
# Hint: Zoomed-in view of just the test period (actual vs predicted)

---
## 9. Limitations of Stock Prediction with ML

While our LSTM may appear to produce reasonable predictions, it is important to understand the fundamental limitations:

**1. The Efficient Market Hypothesis (EMH):**
Stock prices reflect all available information. Any predictable pattern would be quickly arbitraged away by market participants.

**2. The Persistence (Copy) Illusion:**
When the model appears to "predict" well, it is often because tomorrow's price is usually close to today's price. The model may simply be learning to output a value close to its most recent input (a naive lag-1 persistence forecast). On a long price chart the one-day lag is invisible; compare with persistence on the same days and evaluate daily changes (below).

**3. Look-Ahead Bias:**
Using information that was not available at the forecast origin -- for example a scaler fitted on the whole series, a random split that puts later days into training, or features computed from future prices. This notebook fits the scaler on the training days only and splits by target day. One subtle source remains: adjusted prices are recomputed after later splits and dividends, so the early values in the file are not the prices that were quoted at the time.

**4. Non-Stationarity:**
Financial time series are non-stationary -- their statistical properties change over time. A model trained on one regime may fail in another.

**5. External Factors:**
Stock prices are influenced by earnings reports, macroeconomic events, geopolitical developments, and sentiment -- none of which are captured by historical price alone.

**6. Practical Implications:**
Even small prediction errors can lead to large financial losses when leveraged. The real question is not "can we predict direction?" but "can we predict direction better than chance, after accounting for transaction costs?"

In [ ]:
# YOUR CODE HERE
# Hint: Compare your LSTM against a naive baseline on the same test days
# Naive baseline: predict yesterday's close price
# naive_pred = scaler.inverse_transform(X_test[:, -1, :])
# Compute naive_rmse and naive_mae, print comparison

### Evaluate daily changes, not only price levels

A forecast of tomorrow's *level* can look excellent simply because tomorrow's price is close to today's. What a trader or planner needs is the *change*. Convert both forecasts to changes from the last observed close at each origin; persistence then predicts a change of zero. An R^2 near or below zero on changes means no skill, however good the level plot looks. For the direction of the move, compare with always predicting "up".

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. last_close = naive_pred.ravel()   (the last observed close at each origin)
# 2. actual_change = y_actual.ravel() - last_close; lstm_change = y_pred.ravel() - last_close
# 3. Compare r2_score on levels (LSTM vs persistence) with r2_score on changes
#    (persistence predicts a change of zero)
# 4. Direction accuracy: np.mean(np.sign(lstm_change) == np.sign(actual_change)),
#    compared with always predicting "up": np.mean(actual_change > 0)
#    (also check how often the LSTM predicts a rise: np.mean(lstm_change > 0))

---
## 10. Summary

In this notebook we:

1. Loaded Apple stock price data and visualized the close price and its daily changes.
2. Normalized close prices with a `MinMaxScaler` fitted on the training days and created sliding-window sequences.
3. Built a 2-layer LSTM with an FC output head.
4. Trained the model for a fixed number of epochs and plotted the loss curve.
5. Predicted the test days once and plotted predictions against actual prices.
6. Compared against a naive baseline (previous day's close) on the same days, on price levels and on daily changes.
7. Discussed the fundamental limitations of ML-based stock prediction.

**Key takeaways:**
- LSTMs can learn patterns in sequential data, but stock prices are notoriously difficult to predict.
- Always compare against a naive baseline -- many "good" stock models are actually just learning lag-1 persistence.
- Judge the forecast on daily changes: a high R^2 on price levels is mostly the persistence of the price.
- This exercise demonstrates the *mechanics* of applying LSTMs to financial data, not a viable trading strategy.
- Real quantitative finance models incorporate many additional features, risk management, and rigorous backtesting.